# Types of Mechanisms for Subgraph
1) You create two seperate graphs, and in one of the nodes of parent graph, call invoke method of another
child graph.(both have seperate states)
2) subgraph directly added as a node in the parent (shared state)

In [19]:
from langchain_groq import ChatGroq
from langgraph.graph import START,END,StateGraph
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage,BaseMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import add_messages
from dotenv import load_dotenv
from typing import TypedDict,Annotated,Dict,Sequence,List

In [20]:
class parstate(TypedDict):
    inp:str
    anseng:str
    anshin:str

In [21]:
tllm=ChatGroq(model='llama-3.1-8b-instant')
pllm=ChatGroq(model='llama-3.1-8b-instant')

In [22]:
def translate(state:parstate)->parstate:
    prompt=f'''YOu are a helpful assistant. Translate the input to Hindi, Keep it clear, don't change content
        Text: {state['inp']}
    '''
    res=tllm.invoke(prompt).content
    return {'anshin':res}

def answer(state:parstate)->parstate:
    prompt=f''' As a helpful assistant, answer the given question in brief:
     question: {state['inp']} '''
    res=pllm.invoke(prompt).content
    return {'anseng':res}

In [23]:
subgraph=StateGraph(parstate)
subgraph.add_node("translate",translate)
subgraph.add_edge(START,"translate")
subgraph.add_edge("translate",END)

subapp=subgraph.compile()

In [24]:
parentgraph=StateGraph(parstate)
parentgraph.add_node("generator",answer)
parentgraph.add_edge(START,"generator")
parentgraph.add_node("Translator",subapp)   #Here this node is a subgraph as node 
parentgraph.add_edge("generator","Translator")
parentgraph.add_edge("Translator",END)

parapp=parentgraph.compile()

In [25]:
res=parapp.invoke({'inp':"Tell me in brief about India"})
print(res['inp'])
print(res['anseng'])
print(res['anshin'])

Tell me in brief about India
India is a country in South Asia with a rich cultural heritage. Here's a brief overview:

- **Location**: Situated in the Indian subcontinent, bordering Pakistan, China, Nepal, Bhutan, Bangladesh, and Myanmar.
- **Population**: Over 1.38 billion people, making it the second-most populous country in the world.
- **Language**: Official languages are Hindi, English, and 22 other recognized languages.
- **Government**: Federal parliamentary republic with a multi-party system.
- **Economy**: Mixed economy with major sectors in IT, manufacturing, agriculture, and services.
- **Culture**: Diverse, with influences from Hinduism, Islam, Buddhism, and other religions.
- **Cuisine**: Known for spicy dishes like curries, biryani, and tandoori chicken.
- **History**: Ancient civilization with the Indus Valley Civilization (3300 BCE) and the Mughal Empire (1526-1756 CE).
- **Natural resources**: Rich in minerals, including coal, iron, and oil, as well as fertile land and